In [1]:
# download_era5 requires cdsapi + credentials — imported for API reference only
from pyhydra.data_sources.rainfall import download_era5
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import tempfile, os
print('Imports OK')

Imports OK


# ERA5: Reanalysis Data by Copernicus Climate Change Service (C3S)

## What is ERA5?

ERA5 is the fifth-generation reanalysis dataset produced by **ECMWF** under the **Copernicus Climate Change Service (C3S)**.

- **Temporal coverage:** 1950–present
- **Spatial resolution:** ~31 km (0.25°)
- **Temporal resolution:** Hourly

---

## Prerequisites

1. Register at https://cds.climate.copernicus.eu
2. Configure `~/.cdsapirc`:
```ini
url: https://cds.climate.copernicus.eu/api
key: <your-api-key>
```
3. Install: `pip install cdsapi`

> This notebook uses **synthetic demo data** that reproduces the ERA5 format
> without requiring CDS credentials.

---
## ERA5 download — real usage

```python
download_era5(
    folder_out='era5_data/',
    api_key='<your-key>',
    url='https://cds.climate.copernicus.eu/api',
    area=[44, -10, 35, 5],   # [N, W, S, E]
    variables_list=['total_precipitation', '2m_temperature'],
    years=range(2000, 2021),
    months=range(1, 13),
    max_workers=5,
    file_format='netcdf',
    frequency='monthly',
)
```

In [2]:
# === SYNTHETIC DEMO DATA ===
# Creates an xarray Dataset matching ERA5 single-levels format
# Variable: tp (total precipitation, m)

TMPDIR = tempfile.mkdtemp()
rng = np.random.default_rng(42)

times = pd.date_range('2000-01-01', '2020-12-01', freq='MS')  # monthly
lats  = np.arange(44.0, 34.75, -0.25)  # 0.25° grid N→S
lons  = np.arange(-10.0, 5.25,  0.25)  # 0.25° grid W→E

# Synthetic monthly total precipitation (m) — ERA5 stores in metres
months_arr = times.month.values[:, None, None]
seasonal = 0.04 * (1 + np.sin(2 * np.pi * (months_arr - 10) / 12))
tp = (seasonal * rng.uniform(0.5, 1.5, (len(times), len(lats), len(lons)))).astype('float32')

# 2m temperature (K)
t2m_seasonal = 288 + 12 * np.sin(2 * np.pi * (months_arr - 1) / 12)
t2m = (t2m_seasonal + rng.normal(0, 1, (len(times), len(lats), len(lons)))).astype('float32')

ds_era5 = xr.Dataset(
    {
        'tp':  (['time', 'latitude', 'longitude'], tp),
        't2m': (['time', 'latitude', 'longitude'], t2m),
    },
    coords={
        'time':      times,
        'latitude':  lats,
        'longitude': lons,
    }
)
ds_era5['tp'].attrs.update({'long_name': 'Total precipitation', 'units': 'm'})
ds_era5['t2m'].attrs.update({'long_name': '2 metre temperature', 'units': 'K'})

NC_FILE = os.path.join(TMPDIR, 'ERA5_monthly_demo.nc')
ds_era5.to_netcdf(NC_FILE)
print(f'Synthetic ERA5 NetCDF created: {NC_FILE}')
print(ds_era5)

Synthetic ERA5 NetCDF created: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpjn6gh1fg/ERA5_monthly_demo.nc
<xarray.Dataset> Size: 5MB
Dimensions:    (time: 252, latitude: 37, longitude: 61)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2000-01-01 2000-02-01 ... 2020-12-01
  * latitude   (latitude) float64 296B 44.0 43.75 43.5 43.25 ... 35.5 35.25 35.0
  * longitude  (longitude) float64 488B -10.0 -9.75 -9.5 -9.25 ... 4.5 4.75 5.0
Data variables:
    tp         (time, latitude, longitude) float32 2MB 0.1019 ... 0.05707
    t2m        (time, latitude, longitude) float32 2MB 288.3 288.1 ... 282.7


---
## 1. Extract point time series

In [3]:
LAT, LON = 40.4, -3.7   # Madrid

ds = xr.open_dataset(NC_FILE)
pt = ds.sel(latitude=LAT, longitude=LON, method='nearest')
df_era5 = pt[['tp', 't2m']].to_dataframe().reset_index()[['time', 'tp', 't2m']]
df_era5 = df_era5.rename(columns={'time': 'date'})
df_era5['tp_mm'] = df_era5['tp'] * 1000   # convert m → mm
df_era5['t2m_c'] = df_era5['t2m'] - 273.15  # K → °C
df_era5 = df_era5.set_index('date').sort_index()
print(df_era5[['tp_mm', 't2m_c']].describe())
df_era5.head()

            tp_mm       t2m_c
count  252.000000  252.000000
mean    39.666256   14.907264
std     31.868458    8.629343
min      0.000000   -0.041687
25%      9.634757    6.880890
50%     34.757084   15.202301
75%     58.925903   22.866577
max    118.610092   27.927460


,tp,t2m,tp_mm,t2m_c
date,,,,
2000-01-01,0.086479,288.807190,86.479149,15.657196
2000-02-01,0.100247,295.691040,100.247231,22.541046
2000-03-01,0.057823,297.852722,57.822678,24.702728
2000-04-01,0.052791,301.048462,52.791302,27.898468
2000-05-01,0.025754,299.348450,25.753868,26.198456


---
## 2. Visualisation

In [4]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].bar(df_era5.index, df_era5['tp_mm'], color='steelblue', alpha=0.8, width=20)
axes[0].set_ylabel('Monthly precipitation (mm)')
axes[0].set_title(f'ERA5 — Madrid ({LAT}°N, {LON}°E) — synthetic demo', fontsize=13)
axes[0].grid(alpha=0.3, axis='y')

axes[1].plot(df_era5.index, df_era5['t2m_c'], color='tomato', lw=0.8)
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_title('2m Temperature')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ERA5_download.png', dpi=100, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/ERA5_download.png')
ds.close()

Plot saved to /tmp/ERA5_download.png


---
## 3. Annual climatology

In [5]:
# Annual mean precipitation (mm/year)
annual_tp = df_era5['tp_mm'].resample('YE').sum()
annual_t  = df_era5['t2m_c'].resample('YE').mean()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()
ax1.bar(annual_tp.index.year, annual_tp.values, color='steelblue', alpha=0.6, label='Precip (mm)')
ax2.plot(annual_t.index.year, annual_t.values, 'o-', color='tomato', lw=1.5, label='Temp (°C)')
ax1.set_ylabel('Annual precipitation (mm)', color='steelblue')
ax2.set_ylabel('Mean temperature (°C)', color='tomato')
ax1.set_title('ERA5 Annual Climatology — Madrid (synthetic demo)')
ax1.grid(alpha=0.3)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('/tmp/ERA5_climatology.png', dpi=100, bbox_inches='tight')
plt.close()
print('Climatology plot saved to /tmp/ERA5_climatology.png')

Climatology plot saved to /tmp/ERA5_climatology.png
